In [1]:
import africastalking
import pandas as pd
import os
from datetime import datetime, timedelta
import pickle

# Initialize Africa's Talking
africastalking.initialize(
    username='sandbox',
    api_key='atsk_61533322213af52898cc763b2b51a37b924562233cb562d1b1f818c91be2ff2f1406b1fb'
)
voice = africastalking.Voice
sms = africastalking.SMS

# Community language mapping
community_language = {
    'Tamale':           'Dagbani',
    'Bolgatanga':       'Dagbani',
    'Nalerigu':         'Dagbani',
    'Damongo':          'Dagbani',
    'Wa':               'Dagbani',
    'Ho':               'Ewe',
    'Dambai':           'Ewe',
    'Kumasi':           'Twi',
    'Techiman':         'Twi',
    'Sunyani':          'Twi',
    'Goaso':            'Twi',
    'Koforidua':        'Twi',
    'Cape Coast':       'Twi',
    'Sefwi Wiawso':     'Twi',
    'Sekondi-Takoradi': 'Twi',
}

# Verification call messages per language
verification_messages = {
    'Twi': (
        "Mema wo akye. Yɛyɛ AgroAlert Ghana. "
        "Nnawɔtwe bi a atwam no, yɛsomaa wo kra sɛ "
        "obiara bɛba wo mfuom. "
        "Sɛ efiri saa, hyɛ baako. "
        "Sɛ obiara anba, hyɛ mmienu."
    ),
    'Ewe': (
        "Ŋdi na wò. Míle AgroAlert Ghana. "
        "Kɔsiɖa si va yi la, mífia wò be "
        "kpariba ale gɔme na mi ƒe ха̃ame. "
        "Ne emegbe la, zã 1. "
        "Ne mегbe o la, zã 2."
    ),
    'Dagbani': (
        "N daa. Miba AgroAlert Ghana. "
        "Kɔsiɣu bi palli, n kuli n nyɛ i be "
        "kpariba bɛ wari ni i viɛla. "
        "Ni kpariba bia warii, goli 1. "
        "Ni kpariba ka warii, goli 2."
    )
}

# English SMS verification for literate farmers
english_sms = (
    "AgroAlert Ghana Follow-Up:\n"
    "Last week we warned you about drought risk in your area.\n"
    "Did drought actually affect your farm?\n"
    "Reply 1 = YES it did | Reply 2 = NO it did not\n"
    "Your response helps improve our predictions. Thank you."
)

print("Feedback loop initialized!")
print(f"Languages covered: {list(verification_messages.keys())}")

Feedback loop initialized!
Languages covered: ['Twi', 'Ewe', 'Dagbani']


c:\Users\ELITE\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\africastalking\Whatsapp.py:34: UserWarning: Sandbox is currently not available for the Whatsapp service.
  warnings.warn(


In [2]:
# Load predictions and farmer registry
df_predictions = pd.read_csv('C:/Users/ELITE/Documents/AGROALERT/data_processed/predictions.csv')

# Farmer registry with literacy status
farmer_registry = [
    {"name": "Kofi Mensah",    "community": "Ho",       "phone": "+233XXXXXXXXX", "literate": False},
    {"name": "Ama Asante",     "community": "Techiman", "phone": "+233XXXXXXXXX", "literate": True},
    {"name": "Alhassan Baba",  "community": "Tamale",   "phone": "+233XXXXXXXXX", "literate": False},
    {"name": "Efo Agbeko",     "community": "Ho",       "phone": "+233XXXXXXXXX", "literate": True},
]

# Feedback storage
FEEDBACK_FILE = 'C:/Users/ELITE/Documents/AGROALERT/data_processed/farmer_feedback.csv'

if not os.path.exists(FEEDBACK_FILE):
    pd.DataFrame(columns=[
        'timestamp', 'farmer_name', 'phone', 'community',
        'region', 'alert_date', 'ensemble_score',
        'delivery_method', 'language', 'response',
        'drought_confirmed', 'follow_up_sent'
    ]).to_csv(FEEDBACK_FILE, index=False)
    print("Feedback file created!")

def send_verification(farmer, alert_date, score):
    """Send verification follow-up 7 days after alert"""
    community = farmer['community']
    language = community_language.get(community, 'Twi')
    feedback_log = []

    if farmer['literate']:
        # SMS verification in English
        msg = (
            f"AgroAlert Ghana Follow-Up | {community}\n"
            f"Last week we warned you about drought risk "
            f"(score: {score:.2f}).\n"
            f"Did drought actually affect your farm?\n"
            f"Reply 1 = YES it did\n"
            f"Reply 2 = NO it did not\n"
            f"Your reply helps improve our predictions."
        )
        try:
            response = sms.send(msg, [farmer['phone']])
            status = "SMS verification sent"
            print(f"✓ SMS verification → {farmer['name']} ({community}) in English")
        except Exception as e:
            status = f"SMS error: {str(e)[:80]}"
            print(f"✗ SMS error for {farmer['name']}: {e}")

        feedback_log.append({
            'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
            'farmer_name': farmer['name'],
            'phone': farmer['phone'],
            'community': community,
            'alert_date': alert_date,
            'ensemble_score': score,
            'delivery_method': 'SMS',
            'language': 'English',
            'response': None,
            'drought_confirmed': None,
            'follow_up_sent': True
        })

    else:
        # Voice verification in local language
        voice_msg = verification_messages[language]
        print(f"✓ Voice verification queued → {farmer['name']} ({community}) in {language}")
        print(f"  Message: {voice_msg[:80]}...")

        feedback_log.append({
            'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
            'farmer_name': farmer['name'],
            'phone': farmer['phone'],
            'community': community,
            'alert_date': alert_date,
            'ensemble_score': score,
            'delivery_method': f'Voice ({language})',
            'language': language,
            'response': None,
            'drought_confirmed': None,
            'follow_up_sent': True
        })

    return feedback_log

# Run verification for all alerted communities
print("=== Sending Verification Follow-Ups ===\n")
alerted = df_predictions[df_predictions['alert_triggered'] == 1]['community'].unique()
all_feedback = []

for farmer in farmer_registry:
    if farmer['community'] in alerted:
        score = df_predictions[
            df_predictions['community'] == farmer['community']
        ]['ensemble_score'].max()
        alert_date = (datetime.now() - timedelta(days=7)).strftime('%Y-%m-%d')
        logs = send_verification(farmer, alert_date, score)
        all_feedback.extend(logs)

# Save feedback log
df_new = pd.DataFrame(all_feedback)
df_existing = pd.read_csv(FEEDBACK_FILE)
df_combined = pd.concat([df_existing, df_new], ignore_index=True)
df_combined.to_csv(FEEDBACK_FILE, index=False)

print(f"\nVerification log saved — {len(df_new)} follow-ups sent")
print(df_new[['farmer_name','community','delivery_method','language','follow_up_sent']])

Feedback file created!
=== Sending Verification Follow-Ups ===

✓ Voice verification queued → Kofi Mensah (Ho) in Ewe
  Message: Ŋdi na wò. Míle AgroAlert Ghana. Kɔsiɖa si va yi la, mífia wò be kpariba ale gɔm...
✗ SMS error for Ama Asante: Invalid phone number: +233XXXXXXXXX
✓ Voice verification queued → Alhassan Baba (Tamale) in Dagbani
  Message: N daa. Miba AgroAlert Ghana. Kɔsiɣu bi palli, n kuli n nyɛ i be kpariba bɛ wari ...
✗ SMS error for Efo Agbeko: Invalid phone number: +233XXXXXXXXX

Verification log saved — 4 follow-ups sent
     farmer_name community  delivery_method language  follow_up_sent
0    Kofi Mensah        Ho      Voice (Ewe)      Ewe            True
1     Ama Asante  Techiman              SMS  English            True
2  Alhassan Baba    Tamale  Voice (Dagbani)  Dagbani            True
3     Efo Agbeko        Ho              SMS  English            True


In [3]:
# Simulate farmer keypress responses coming back
# In real deployment these come from Africa's Talking webhook
# 1 = Yes drought happened | 2 = No drought did not happen

simulated_responses = [
    {"farmer_name": "Kofi Mensah",   "phone": "+233XXXXXXXXX", "keypress": "1", "community": "Ho"},
    {"farmer_name": "Ama Asante",    "phone": "+233XXXXXXXXX", "keypress": "1", "community": "Techiman"},
    {"farmer_name": "Alhassan Baba", "phone": "+233XXXXXXXXX", "keypress": "2", "community": "Tamale"},
    {"farmer_name": "Efo Agbeko",    "phone": "+233XXXXXXXXX", "keypress": "1", "community": "Ho"},
]

# Load and update feedback file with responses
df_feedback = pd.read_csv(FEEDBACK_FILE)

for response in simulated_responses:
    mask = df_feedback['farmer_name'] == response['farmer_name']
    df_feedback.loc[mask, 'response'] = response['keypress']
    df_feedback.loc[mask, 'drought_confirmed'] = (response['keypress'] == '1')

df_feedback.to_csv(FEEDBACK_FILE, index=False)

print("=== Farmer Responses Recorded ===\n")
print(df_feedback[['farmer_name','community','delivery_method',
                    'ensemble_score','response','drought_confirmed']])

# Count ground truth
confirmed = df_feedback['drought_confirmed'].sum()
denied = (df_feedback['drought_confirmed'] == False).sum()
print(f"\nDrought confirmed by farmers: {int(confirmed)}")
print(f"Drought denied by farmers: {int(denied)}")
print(f"Ground truth response rate: {len(df_feedback[df_feedback['response'].notna()])}/{len(df_feedback)}")

TypeError: Invalid value '1' for dtype 'float64'

In [5]:
# Simulate farmer keypress responses
simulated_responses = [
    {"farmer_name": "Kofi Mensah",   "keypress": "1"},
    {"farmer_name": "Ama Asante",    "keypress": "1"},
    {"farmer_name": "Alhassan Baba", "keypress": "2"},
    {"farmer_name": "Efo Agbeko",    "keypress": "1"},
]

df_feedback = pd.read_csv(FEEDBACK_FILE)

# Fix dtype before assignment
df_feedback['response'] = df_feedback['response'].astype(object)
df_feedback['drought_confirmed'] = df_feedback['drought_confirmed'].astype(object)

for r in simulated_responses:
    mask = df_feedback['farmer_name'] == r['farmer_name']
    df_feedback.loc[mask, 'response'] = r['keypress']
    df_feedback.loc[mask, 'drought_confirmed'] = (r['keypress'] == '1')

df_feedback.to_csv(FEEDBACK_FILE, index=False)

confirmed = df_feedback[df_feedback['drought_confirmed'] == True].shape[0]
denied = df_feedback[df_feedback['drought_confirmed'] == False].shape[0]

print("=== Farmer Responses Recorded ===\n")
print(df_feedback[['farmer_name','community','delivery_method',
                    'ensemble_score','response','drought_confirmed']])
print(f"\nDrought confirmed by farmers: {confirmed}")
print(f"Drought denied by farmers: {denied}")
print(f"Response rate: {confirmed + denied}/{len(df_feedback)}")

=== Farmer Responses Recorded ===

     farmer_name community  delivery_method  ensemble_score response  \
0    Kofi Mensah        Ho      Voice (Ewe)        0.878798        1   
1     Ama Asante  Techiman              SMS        0.878429        1   
2  Alhassan Baba    Tamale  Voice (Dagbani)        0.878437        2   
3     Efo Agbeko        Ho              SMS        0.878798        1   

  drought_confirmed  
0              True  
1              True  
2             False  
3              True  

Drought confirmed by farmers: 3
Drought denied by farmers: 1
Response rate: 4/4


In [6]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
import pickle

# Load master dataset
df_master = pd.read_csv('C:/Users/ELITE/Documents/AGROALERT/data_processed/master_dataset.csv')

# Load feedback — only confirmed responses
df_fb = pd.read_csv(FEEDBACK_FILE)
df_fb = df_fb[df_fb['response'].notna()].copy()

print(f"Ground truth feedback records: {len(df_fb)}")

# Match feedback to master dataset rows and update labels
df_master['date'] = pd.to_datetime(df_master['date'])
updated = 0

for _, row in df_fb.iterrows():
    community = row['community']
    confirmed = row['drought_confirmed']
    # Update the most recent week's label for this community
    mask = (df_master['community'] == community)
    last_idx = df_master[mask].index[-1]
    old_label = df_master.loc[last_idx, 'drought_label']
    new_label = 1 if confirmed else 0
    df_master.loc[last_idx, 'drought_label'] = new_label
    if old_label != new_label:
        updated += 1
        print(f"  Updated {community}: label {old_label} → {new_label} (farmer confirmed: {confirmed})")

print(f"\n{updated} labels updated from farmer ground truth")

# Retrain model with updated labels
le_community = LabelEncoder()
le_region = LabelEncoder()
df_master['community_enc'] = le_community.fit_transform(df_master['community'])
df_master['region_enc'] = le_region.fit_transform(df_master['region'])

features = ['community_enc','region_enc','ndvi','rainfall_mm',
            'temp_max','temp_min','humidity','et0',
            'lst_celsius','ndvi_anomaly','rainfall_deficit',
            'water_balance','spei_proxy']

X = df_master[features]
y = df_master['drought_label']

rf_model = RandomForestClassifier(
    n_estimators=200, max_depth=10,
    min_samples_split=5, class_weight='balanced',
    random_state=42
)
rf_model.fit(X, y)

# Save updated model
with open('C:/Users/ELITE/Documents/AGROALERT/src_model/rf_model.pkl','wb') as f:
    pickle.dump(rf_model, f)
with open('C:/Users/ELITE/Documents/AGROALERT/src_model/le_community.pkl','wb') as f:
    pickle.dump(le_community, f)
with open('C:/Users/ELITE/Documents/AGROALERT/src_model/le_region.pkl','wb') as f:
    pickle.dump(le_region, f)

# Save updated master dataset
df_master.to_csv('C:/Users/ELITE/Documents/AGROALERT/data_processed/master_dataset.csv', index=False)

print(f"\nModel retrained with farmer ground truth!")
print(f"Total drought labels in dataset: {y.sum()}")
print(f"Model saved successfully.")

Ground truth feedback records: 4
  Updated Techiman: label 0 → 1 (farmer confirmed: True)

1 labels updated from farmer ground truth

Model retrained with farmer ground truth!
Total drought labels in dataset: 29
Model saved successfully.
